# Playground Series S6E3 - Telco Churn: XGBoost with Pseudo-Labeling

**Competition:** [Playground Series S6E3](https://www.kaggle.com/competitions/playground-series-s6e3)  
**Problem:** binary classification - predict whether each telco customer will *churn* (cancel service).  
**Data:** a synthetic competition train/test generated from the classic IBM Telco Customer Churn table, plus the original IBM table used as extra training signal.  
**Approach:** heavy tabular feature engineering -> gradient-boosted trees (XGBoost) -> a conservative **pseudo-labeling** loop that adds only the most confident test predictions back into training.  
**Author:** Lorenzo Scaturchio

---

This notebook is tuned for **competition score movement** on the AUC leaderboard, but every step is explained so the pipeline stays readable and reproducible.

## 1. Objective

The **goal** is to rank customers by their probability of churning. The competition is scored with **ROC AUC**, so we care about *ordering* customers correctly far more than about the absolute calibration of any single probability. That choice of metric shapes everything downstream: we keep raw probabilities (never hard 0/1 labels) in the submission, and we judge every modelling decision with held-out AUC.

### Why pseudo-labeling on a tabular Playground task?

Playground test sets are large and drawn from the *same generator* as the train set, so the test feature distribution is trustworthy. **Pseudo-labeling** exploits that: we train a model, score the test rows, and fold the most confident predictions back in as extra (noisy) training labels. The **hypothesis** is that the easy, high-confidence test rows teach the model a little more about the decision boundary and shrink variance, *because* they expand the effective sample size in regions the model is already sure about.

The obvious **trade-off** is confirmation bias: if we trust the model too much, we simply re-feed its own mistakes. We manage that risk with three guards introduced later - a strict confidence gate, down-weighted pseudo rows, and a held-out fold that the pseudo labels never touch.

## 2. Setup and Reproducibility

A single `RANDOM_STATE` constant flows into every stochastic component - the NumPy global seed, the train/validation split, the cross-validation folds, the target encoder, and XGBoost itself. Pinning one seed everywhere means the AUC numbers below are reproducible run-to-run, which is essential when we later compare the model *before* and *after* pseudo-labeling and need the difference to be signal rather than seed noise.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import TargetEncoder
import xgboost as xgb

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid', context='notebook')

# Single source of randomness for the whole notebook.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'Environment ready. RANDOM_STATE={RANDOM_STATE}')

## 3. Dataset and Data Loading

Three tables feed the pipeline:

| Table | Role | Source |
|---|---|---|
| `train.csv` | labelled competition rows | Playground S6E3 |
| `test.csv` | unlabelled rows we must score | Playground S6E3 |
| IBM Telco churn CSV | extra labelled rows + target statistics | original public dataset |

The loader searches a few canonical `/kaggle/input` locations and then falls back to a recursive glob, so the notebook keeps running even if the dataset is mounted under a slightly different slug.

In [ ]:
def find_input_file(filename: str) -> Path | None:
    candidates = [
        Path('/kaggle/input/playground-series-s6e3') / filename,
        Path('/kaggle/input/competitions/playground-series-s6e3') / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob(filename))
        if matches:
            return matches[0]
    return None

def find_original_telco_file() -> Path | None:
    candidates = [
        Path('/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
        Path('/kaggle/input/wa-fnusec-telcocustomerchurn/WA_Fn-UseC_-Telco-Customer-Churn.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    input_root = Path('/kaggle/input')
    if input_root.exists():
        matches = sorted(input_root.rglob('WA_Fn-UseC_-Telco-Customer-Churn.csv'))
        if matches:
            return matches[0]
    return None

train_path = find_input_file('train.csv')
test_path = find_input_file('test.csv')
orig_path = find_original_telco_file()

if train_path is None or test_path is None or orig_path is None:
    raise FileNotFoundError('Required competition or original telco files were not found under /kaggle/input.')

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
orig = pd.read_csv(orig_path)

print(f'train: {train.shape}')
print(f'test : {test.shape}')
print(f'orig : {orig.shape}')
print(f'competition train path: {train_path}')
print(f'original telco path  : {orig_path}')

## 4. Exploratory Data Analysis

Before any modelling we sanity-check the schema, the target balance, and data quality. Telco churn tables are notorious for one quirk: `TotalCharges` is stored as a string and contains blank values for brand-new customers (tenure 0), which silently turns the column into `object` dtype. We confirm that here and handle it explicitly during feature engineering.

In [ ]:
# Schema and per-column dtype / missingness snapshot for the competition train set.
schema = pd.DataFrame({
    'dtype': train.dtypes.astype(str),
    'n_unique': train.nunique(),
    'n_missing': train.isna().sum(),
    'pct_missing': (train.isna().mean() * 100).round(2),
})
print(f'Columns: {train.shape[1]} | Rows: {train.shape[0]:,}')
print(f"TotalCharges raw dtype: {train['TotalCharges'].dtype} "
      f"(blank-string rows: {(train['TotalCharges'].astype(str).str.strip() == '').sum()})")
schema.sort_values('n_missing', ascending=False).head(20)

### Target balance

Class balance drives how we read AUC and whether we need to worry about a skewed prior. The bar chart below shows the churn rate in the competition train set next to the original IBM table. If the two rates are close, the original rows are a reasonable source of extra signal; a large gap would be a **caveat** that the auxiliary labels carry distribution shift.

In [ ]:
# Normalize both targets to 0/1 just for the plot (the pipeline does this robustly later).
def _churn01(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0}).fillna(s)

train_rate = _churn01(train['Churn']).astype(float).mean()
orig_rate = _churn01(orig['Churn']).astype(float).mean()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
counts = _churn01(train['Churn']).value_counts().sort_index()
ax[0].bar(['No churn (0)', 'Churn (1)'], counts.values, color=['#4C72B0', '#C44E52'])
ax[0].set_title('Competition train: churn class counts')
ax[0].set_ylabel('customers')
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f'{v:,}', ha='center', va='bottom')

ax[1].bar(['Competition train', 'Original IBM'], [train_rate, orig_rate], color=['#4C72B0', '#55A868'])
ax[1].set_title('Churn rate: competition vs original')
ax[1].set_ylabel('P(churn)')
ax[1].set_ylim(0, max(train_rate, orig_rate) * 1.4)
for i, v in enumerate([train_rate, orig_rate]):
    ax[1].text(i, v, f'{v:.1%}', ha='center', va='bottom')
plt.tight_layout()
plt.show()
print(f'train churn rate={train_rate:.3f} | orig churn rate={orig_rate:.3f}')

### Key feature distributions

Three numeric columns carry most of the churn signal in this domain: `tenure` (months as a customer), `MonthlyCharges`, and `TotalCharges`. We plot each one split by churn status. The recurring **observation** in telco data - and a useful prior for feature design - is that churn concentrates among *short-tenure, high-monthly-charge* customers, while long-tenured customers almost never leave.

In [ ]:
# Distribution of the three core numeric drivers, split by churn.
plot_df = train.copy()
plot_df['TotalCharges'] = pd.to_numeric(plot_df['TotalCharges'], errors='coerce')
plot_df['churn01'] = _churn01(plot_df['Churn']).astype(int)

numeric_drivers = ['tenure', 'MonthlyCharges', 'TotalCharges']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_drivers):
    for label, color in [(0, '#4C72B0'), (1, '#C44E52')]:
        sns.kdeplot(
            data=plot_df[plot_df['churn01'] == label],
            x=col, ax=ax, fill=True, alpha=0.35, color=color,
            label=('churn' if label else 'stay'), warn_singular=False,
        )
    ax.set_title(f'{col} by churn status')
    ax.legend()
plt.tight_layout()
plt.show()

### Churn rate by contract type

`Contract` is the single strongest categorical predictor in telco churn. Month-to-month customers have nothing locking them in, so they churn at a far higher rate than one- or two-year contract holders. Confirming this gives us confidence that the categorical-encoding machinery downstream is working on real signal rather than noise.

In [ ]:
# Churn rate per contract type with sample sizes annotated.
by_contract = (
    plot_df.groupby('Contract')['churn01']
    .agg(['mean', 'count'])
    .sort_values('mean', ascending=False)
)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(by_contract.index, by_contract['mean'], color='#8172B3')
ax.set_title('Churn rate by contract type')
ax.set_ylabel('P(churn)')
ax.set_ylim(0, by_contract['mean'].max() * 1.25)
for bar, (rate, n) in zip(bars, by_contract[['mean', 'count']].values):
    ax.text(bar.get_x() + bar.get_width() / 2, rate, f'{rate:.1%}\n(n={int(n):,})', ha='center', va='bottom')
plt.tight_layout()
plt.show()
by_contract

## 5. Method - Feature Engineering

The modelling **approach** has three layers, and feature engineering is the first. We expand the raw schema into a wide family of tabular features that consistently move tree-model AUC on telco data:

- **Charge ratios and residuals** - `MonthlyCharges / TotalCharges`, average monthly spend, and the   `charges_deviation` between observed `TotalCharges` and `tenure * MonthlyCharges`. The deviation is   a cheap data-quality / loyalty signal.
- **Service counts** - how many add-on services (security, backup, streaming, ...) a customer holds.   Bundled customers are stickier.
- **Binned numerics** - coarse `tenure` / charge buckets that let the trees split on regime, not just   raw value.
- **Frequency and original-target encodings** - each category is mapped to its frequency and to its   mean churn rate *in the original IBM table*, which injects out-of-fold signal without leaking the   competition labels.
- **Rank / z-score features** vs churner and non-churner reference distributions of `TotalCharges`.

The `TotalCharges` blanks we found in EDA are filled with `MonthlyCharges * tenure`, the natural estimate for a customer with no billing history yet.

In [ ]:
def concat_feature_block(df: pd.DataFrame, updates: dict[str, object]) -> pd.DataFrame:
    if not updates:
        return df
    return pd.concat([df, pd.DataFrame(updates, index=df.index)], axis=1).copy()

def prepare_target(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().map({'yes': 1, 'no': 0}).fillna(series).astype(int)

def rank_against_reference(values: pd.Series, reference: pd.Series) -> pd.Series:
    ref = np.sort(reference.to_numpy(dtype=float))
    ranked = np.searchsorted(ref, values.to_numpy(dtype=float), side='right') / max(len(ref), 1)
    return pd.Series(ranked, index=values.index, dtype=float)

def advanced_feature_frames(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    orig_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[str], list[str]]:
    target = 'Churn'
    train = train_df.copy()
    test = test_df.copy()
    orig = orig_df.copy()
    train[target] = prepare_target(train[target])
    orig[target] = prepare_target(orig[target])

    base_cat_cols = [
        'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
        'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
    ]
    service_cols = [
        'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
        'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    ]
    pair_cols = [
        ('Contract', 'InternetService'),
        ('Contract', 'PaymentMethod'),
        ('InternetService', 'PaymentMethod'),
        ('PaperlessBilling', 'PaymentMethod'),
    ]

    for df in (train, test, orig):
        df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
        df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'])
        df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)
        for col in base_cat_cols:
            df[col] = df[col].fillna('Missing').astype(str)

        df['charges_deviation'] = df['TotalCharges'] - (df['MonthlyCharges'] * df['tenure'])
        df['abs_charges_dev'] = np.abs(df['charges_deviation'])
        df['monthly_to_total_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1.0)
        df['total_to_monthly_ratio'] = df['TotalCharges'] / (df['MonthlyCharges'] + 1.0)
        df['avg_monthly_charges'] = df['TotalCharges'] / np.maximum(df['tenure'], 1)
        df['tenure_x_monthly'] = df['tenure'] * df['MonthlyCharges']
        df['tenure_x_total'] = df['tenure'] * df['TotalCharges']
        yes_count = sum(df[col].eq('Yes') for col in service_cols)
        no_count = sum(df[col].eq('No') for col in service_cols)
        other_count = len(service_cols) - yes_count - no_count
        df['service_yes_count'] = yes_count.astype(int)
        df['service_no_count'] = no_count.astype(int)
        df['service_other_count'] = other_count.astype(int)
        df['service_count'] = (yes_count + (0.5 * other_count)).astype(float)
        df['has_internet'] = (~df['InternetService'].eq('No')).astype(int)
        df['has_phone'] = df['PhoneService'].eq('Yes').astype(int)

    bin_specs = {
        'tenure_bin': [-1, 0, 6, 12, 24, 48, 72, 200],
        'MonthlyCharges_bin': [0, 20, 40, 60, 80, 100, 200],
        'TotalCharges_bin': [-1, 0, 250, 1000, 2500, 5000, 10000],
    }
    for name, bins in bin_specs.items():
        source = name.replace('_bin', '')
        train[name] = pd.cut(train[source], bins=bins, labels=False, include_lowest=True).fillna(-1).astype(int).astype(str)
        test[name] = pd.cut(test[source], bins=bins, labels=False, include_lowest=True).fillna(-1).astype(int).astype(str)
        orig[name] = pd.cut(orig[source], bins=bins, labels=False, include_lowest=True).fillna(-1).astype(int).astype(str)

    indicator_triplets = [
        ('ISYES_', 'Yes'),
        ('ISNO_', 'No'),
    ]
    for prefix, matcher in indicator_triplets:
        train = concat_feature_block(train, {f'{prefix}{col}': train[col].eq(matcher).astype(int) for col in service_cols})
        test = concat_feature_block(test, {f'{prefix}{col}': test[col].eq(matcher).astype(int) for col in service_cols})
        orig = concat_feature_block(orig, {f'{prefix}{col}': orig[col].eq(matcher).astype(int) for col in service_cols})
    train = concat_feature_block(train, {f'ISOTHER_{col}': (~train[col].isin(['Yes', 'No'])).astype(int) for col in service_cols})
    test = concat_feature_block(test, {f'ISOTHER_{col}': (~test[col].isin(['Yes', 'No'])).astype(int) for col in service_cols})
    orig = concat_feature_block(orig, {f'ISOTHER_{col}': (~orig[col].isin(['Yes', 'No'])).astype(int) for col in service_cols})

    pair_feature_names = []
    for left, right in pair_cols:
        feature_name = f'BG_{left}_{right}'
        pair_feature_names.append(feature_name)
        train[feature_name] = train[left].astype(str) + '__' + train[right].astype(str)
        test[feature_name] = test[left].astype(str) + '__' + test[right].astype(str)
        orig[feature_name] = orig[left].astype(str) + '__' + orig[right].astype(str)

    enriched_cat_cols = base_cat_cols + ['tenure_bin', 'MonthlyCharges_bin', 'TotalCharges_bin'] + pair_feature_names
    orig_target_mean = float(orig[target].mean())
    combined_for_freq = pd.concat([train[enriched_cat_cols], orig[enriched_cat_cols]], axis=0, ignore_index=True)
    for col in enriched_cat_cols:
        freq = combined_for_freq[col].value_counts(dropna=False, normalize=True)
        train[f'FREQ_{col}'] = train[col].map(freq).fillna(0).astype(float)
        test[f'FREQ_{col}'] = test[col].map(freq).fillna(0).astype(float)
        orig[f'FREQ_{col}'] = orig[col].map(freq).fillna(0).astype(float)

        mapping = orig.groupby(col, observed=False)[target].mean()
        train[f'ORIG_proba_{col}'] = train[col].map(mapping).fillna(orig_target_mean).astype(float)
        test[f'ORIG_proba_{col}'] = test[col].map(mapping).fillna(orig_target_mean).astype(float)
        orig[f'ORIG_proba_{col}'] = orig[col].map(mapping).fillna(orig_target_mean).astype(float)

    churn_tc = train.loc[train[target] == 1, 'TotalCharges']
    non_tc = train.loc[train[target] == 0, 'TotalCharges']
    mc_mean_by_is = train.groupby('InternetService', observed=False)['MonthlyCharges'].mean().to_dict()
    is_rank_lookup = train.assign(_rank=train.groupby('InternetService', observed=False)['TotalCharges'].rank(pct=True)).groupby('InternetService', observed=False)['_rank'].mean().to_dict()
    contract_rank_lookup = train.assign(_rank=train.groupby('Contract', observed=False)['TotalCharges'].rank(pct=True)).groupby('Contract', observed=False)['_rank'].mean().to_dict()

    for df in (train, test, orig):
        df['pctrank_orig_TC'] = rank_against_reference(df['TotalCharges'], orig['TotalCharges'])
        df['pctrank_churner_TC'] = rank_against_reference(df['TotalCharges'], churn_tc)
        df['pctrank_nonchurner_TC'] = rank_against_reference(df['TotalCharges'], non_tc)
        df['zscore_churn_gap_TC'] = df['pctrank_churner_TC'] - df['pctrank_nonchurner_TC']
        df['zscore_nonchurner_TC'] = (df['TotalCharges'] - float(non_tc.mean())) / (float(non_tc.std()) + 1e-6)
        df['pctrank_churn_gap_TC'] = df['pctrank_churner_TC'] - df['pctrank_nonchurner_TC']
        df['resid_IS_MC'] = df['MonthlyCharges'] - df['InternetService'].map(mc_mean_by_is).fillna(float(train['MonthlyCharges'].mean()))
        df['cond_pctrank_IS_TC'] = df['InternetService'].map(is_rank_lookup).fillna(0.5).astype(float)
        df['cond_pctrank_C_TC'] = df['Contract'].map(contract_rank_lookup).fillna(0.5).astype(float)

    for df in (train, test, orig):
        for col in enriched_cat_cols:
            df[col] = df[col].astype(str).astype('category')

    indicator_cols = [c for c in train.columns if c.startswith(('ISYES_', 'ISNO_', 'ISOTHER_'))]
    freq_cols = [f'FREQ_{col}' for col in enriched_cat_cols]
    proba_cols = [f'ORIG_proba_{col}' for col in enriched_cat_cols]
    num_cols = [
        'tenure', 'MonthlyCharges', 'TotalCharges', 'charges_deviation', 'abs_charges_dev',
        'monthly_to_total_ratio', 'total_to_monthly_ratio', 'avg_monthly_charges',
        'tenure_x_monthly', 'tenure_x_total', 'service_yes_count', 'service_no_count',
        'service_other_count', 'service_count', 'has_internet', 'has_phone',
        'pctrank_orig_TC', 'pctrank_churner_TC', 'pctrank_nonchurner_TC',
        'zscore_churn_gap_TC', 'zscore_nonchurner_TC', 'pctrank_churn_gap_TC',
        'resid_IS_MC', 'cond_pctrank_IS_TC', 'cond_pctrank_C_TC',
    ] + indicator_cols + freq_cols + proba_cols
    feature_cols = num_cols + enriched_cat_cols
    te_cols = enriched_cat_cols.copy()
    drop_raw_cols = te_cols.copy()
    return train, test, feature_cols, te_cols, drop_raw_cols

train_frame, test_frame, feature_cols, te_cols, drop_raw_cols = advanced_feature_frames(train, test, orig)
print({'feature_count': len(feature_cols), 'te_cols': len(te_cols), 'train_rows': len(train_frame), 'test_rows': len(test_frame)})
train_frame[feature_cols].head()

## 6. Method - XGBoost Model and the Pseudo-Labeling Loop

The second layer is the model. `fit_encoded_xgb` wraps two leakage-safe encoders around an XGBoost classifier:

1. **Out-of-fold group statistics** (`std/min/max` of the target per category) computed with an inner    3-fold `StratifiedKFold` so a row never sees its own label in its encoding.
2. A scikit-learn **`TargetEncoder`** (smoothed, cross-fitted) for the categorical mean encoding.

XGBoost itself runs with `tree_method='hist'`, native categorical support, a low learning rate (`0.03`) with early stopping on validation AUC, and explicit `random_state=RANDOM_STATE`.

### The pseudo-labeling loop, step by step

1. **Train** the base model on the 80% train split.
2. **Predict** churn probability for every test row.
3. **Gate** on confidence: keep only rows in the extreme tails - below the 8th percentile *and* under    `1 - 0.92`, or above the 92nd percentile *and* over `0.92`. The dual quantile-and-absolute rule    means we never accept a row the model is merely *relatively* confident about.
4. **Down-weight** every accepted pseudo row to `0.2-0.4` of a real row, so genuine labels always    dominate the gradient.
5. **Retrain** once on the augmented matrix and re-measure AUC on the *same untouched validation fold*.

### The confidence-threshold trade-off

A **lower** threshold harvests more pseudo rows (bigger sample) but admits more label noise; a **higher** threshold keeps labels cleaner but adds little new information. We sit deliberately on the conservative side and additionally require a minimum harvest size before bothering to retrain, *because* a handful of pseudo rows cannot justify the extra variance of a second fit. This is the core **trade-off** of the whole technique.

In [ ]:
def pseudo_label_mask(predictions: np.ndarray, lower_quantile: float = 0.08, upper_quantile: float = 0.92, absolute_confidence: float = 0.92) -> np.ndarray:
    lower_threshold = min(float(np.quantile(predictions, lower_quantile)), 1.0 - absolute_confidence)
    upper_threshold = max(float(np.quantile(predictions, upper_quantile)), absolute_confidence)
    return (predictions <= lower_threshold) | (predictions >= upper_threshold)

def pseudo_label_weights(predictions: np.ndarray) -> np.ndarray:
    confidence = np.abs(predictions - 0.5) * 2.0
    return np.clip(0.15 + (0.25 * confidence), 0.2, 0.4)

def fit_encoded_xgb(
    x_train: pd.DataFrame,
    y_train: np.ndarray,
    x_valid: pd.DataFrame,
    y_valid: np.ndarray,
    x_test: pd.DataFrame,
    te_cols_local: list[str],
    drop_cols_local: list[str],
    sample_weight: np.ndarray | None = None,
) -> tuple[xgb.XGBClassifier, np.ndarray, np.ndarray]:
    stats = ['std', 'min', 'max']
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    te_stat_cols = [f'TE1_{col}_{stat}' for col in te_cols_local for stat in stats]
    x_train = concat_feature_block(x_train, {name: np.nan for name in te_stat_cols})

    for inner_train_idx, inner_valid_idx in inner_cv.split(x_train, y_train):
        x_inner_train = x_train.loc[inner_train_idx, feature_cols + ['Churn']].copy()
        x_inner_valid = x_train.loc[inner_valid_idx, feature_cols].copy()
        for col in te_cols_local:
            grouped = x_inner_train.groupby(col, observed=False)['Churn'].agg(stats)
            grouped.columns = [f'TE1_{col}_{stat}' for stat in stats]
            x_inner_valid = x_inner_valid.merge(grouped, on=col, how='left')
            for name in grouped.columns:
                x_train.loc[inner_valid_idx, name] = x_inner_valid[name].to_numpy(dtype='float32')

    for col in te_cols_local:
        grouped = x_train.groupby(col, observed=False)['Churn'].agg(stats)
        grouped.columns = [f'TE1_{col}_{stat}' for stat in stats]
        x_valid = x_valid.merge(grouped.astype('float32'), on=col, how='left')
        x_test = x_test.merge(grouped.astype('float32'), on=col, how='left')
        for name in grouped.columns:
            x_train[name] = x_train[name].fillna(0).astype('float32')
            x_valid[name] = x_valid[name].fillna(0).astype('float32')
            x_test[name] = x_test[name].fillna(0).astype('float32')

    mean_encoder = TargetEncoder(cv=3, shuffle=True, smooth='auto', target_type='binary', random_state=RANDOM_STATE)
    mean_cols = [f'TE_{col}' for col in te_cols_local]
    x_train = pd.concat([x_train, pd.DataFrame(mean_encoder.fit_transform(x_train[te_cols_local], y_train), columns=mean_cols, index=x_train.index)], axis=1).copy()
    x_valid = pd.concat([x_valid, pd.DataFrame(mean_encoder.transform(x_valid[te_cols_local]), columns=mean_cols, index=x_valid.index)], axis=1).copy()
    x_test = pd.concat([x_test, pd.DataFrame(mean_encoder.transform(x_test[te_cols_local]), columns=mean_cols, index=x_test.index)], axis=1).copy()

    for df in (x_train, x_valid, x_test):
        for col in te_cols_local:
            df[col] = df[col].astype(str).astype('category')
        df.drop(columns=drop_cols_local, inplace=True)
    x_train = x_train.drop(columns=['Churn'])

    params = {
        'n_estimators': 2400,
        'learning_rate': 0.03,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'min_child_weight': 5,
        'reg_alpha': 0.1,
        'reg_lambda': 1.0,
        'gamma': 0.05,
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'enable_categorical': True,
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbosity': 0,
        'early_stopping_rounds': 80,
    }
    model = xgb.XGBClassifier(**params)
    fit_kwargs = {'eval_set': [(x_valid, y_valid)], 'verbose': False}
    if sample_weight is not None:
        fit_kwargs['sample_weight'] = sample_weight
    model.fit(x_train, y_train, **fit_kwargs)
    valid_pred = model.predict_proba(x_valid)[:, 1]
    test_pred = model.predict_proba(x_test)[:, 1]
    return model, valid_pred, test_pred

y = train_frame['Churn'].to_numpy()
train_idx, valid_idx = train_test_split(np.arange(len(train_frame)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
x_train = train_frame.iloc[train_idx][feature_cols + ['Churn']].reset_index(drop=True).copy()
y_train = y[train_idx]
x_valid = train_frame.iloc[valid_idx][feature_cols].reset_index(drop=True).copy()
y_valid = y[valid_idx]
x_test = test_frame[feature_cols].reset_index(drop=True).copy()

base_model, base_valid_pred, base_test_pred = fit_encoded_xgb(
    x_train.copy(),
    y_train,
    x_valid.copy(),
    y_valid,
    x_test.copy(),
    te_cols,
    drop_raw_cols,
)
base_auc = roc_auc_score(y_valid, base_valid_pred)
pseudo_mask = pseudo_label_mask(base_test_pred)
pseudo_count = int(pseudo_mask.sum())

if pseudo_count >= max(2000, len(base_test_pred) // 50):
    pseudo_x = x_test.loc[pseudo_mask].copy()
    pseudo_y = (base_test_pred[pseudo_mask] >= 0.5).astype(int)
    pseudo_w = pseudo_label_weights(base_test_pred[pseudo_mask])
    augmented_x = pd.concat([x_train, pseudo_x], axis=0, ignore_index=True).copy()
    augmented_y = np.concatenate([y_train, pseudo_y])
    augmented_x['Churn'] = augmented_y
    sample_weight = np.concatenate([np.ones(len(y_train), dtype=float), pseudo_w])
    pseudo_model, pseudo_valid_pred, _ = fit_encoded_xgb(
        augmented_x.copy(),
        augmented_y,
        x_valid.copy(),
        y_valid,
        x_test.copy(),
        te_cols,
        drop_raw_cols,
        sample_weight=sample_weight,
    )
    pseudo_auc = roc_auc_score(y_valid, pseudo_valid_pred)
else:
    pseudo_model = base_model
    pseudo_valid_pred = base_valid_pred
    pseudo_auc = base_auc

print({'base_holdout_auc': round(float(base_auc), 5), 'pseudo_holdout_auc': round(float(pseudo_auc), 5), 'pseudo_rows': pseudo_count})

## 7. Evaluation - Validation Metrics

Everything is judged on the held-out 20% fold that the pseudo labels never entered. The third layer of the approach is honest measurement: we compare base vs pseudo-labeled **AUC**, draw the ROC curve, and inspect feature importance. The first cell visualises the confidence gate so we can *see* exactly how much of the test set was harvested and where the threshold cut.

In [ ]:
# Visualise the confidence gate: which test predictions become pseudo-labels.
lower_t = min(float(np.quantile(base_test_pred, 0.08)), 1.0 - 0.92)
upper_t = max(float(np.quantile(base_test_pred, 0.92)), 0.92)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(base_test_pred, bins=60, color='#cccccc', edgecolor='white')
ax.axvline(lower_t, color='#4C72B0', linestyle='--', label=f'lower gate = {lower_t:.3f}')
ax.axvline(upper_t, color='#C44E52', linestyle='--', label=f'upper gate = {upper_t:.3f}')
ax.axvspan(0, lower_t, color='#4C72B0', alpha=0.12)
ax.axvspan(upper_t, 1, color='#C44E52', alpha=0.12)
ax.set_title(f'Base test-prediction distribution and confidence gate\n'
             f'{pseudo_count:,} of {len(base_test_pred):,} rows kept as pseudo-labels')
ax.set_xlabel('predicted P(churn)')
ax.set_ylabel('test rows')
ax.legend()
plt.tight_layout()
plt.show()

### Base vs pseudo-labeled AUC

The bar chart puts the two held-out AUC numbers side by side. The interesting question is not just whether the second bar is taller, but by *how much* relative to typical seed-level noise on this fold. A delta of a few ten-thousandths is consistent with pseudo-labeling helping at the margin; a negative delta would be our cue to tighten the gate.

In [ ]:
# Side-by-side comparison of held-out AUC, base vs pseudo-labeled.
auc_delta = float(pseudo_auc) - float(base_auc)
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(['Base XGBoost', 'Pseudo-labeled'], [base_auc, pseudo_auc],
              color=['#4C72B0', '#55A868'])
lo = min(base_auc, pseudo_auc)
hi = max(base_auc, pseudo_auc)
ax.set_ylim(lo - 0.01, hi + 0.005)
ax.set_ylabel('held-out ROC AUC')
ax.set_title(f'Validation AUC before vs after pseudo-labeling\n(delta = {auc_delta:+.5f})')
for bar, v in zip(bars, [base_auc, pseudo_auc]):
    ax.text(bar.get_x() + bar.get_width() / 2, v, f'{v:.5f}', ha='center', va='bottom')
plt.tight_layout()
plt.show()
print(f'base_auc={base_auc:.5f}  pseudo_auc={pseudo_auc:.5f}  delta={auc_delta:+.5f}')

### ROC curve

The ROC curve shows the full sensitivity/specificity trade-off for both models on the same fold. Because the metric *is* the area under this curve, overlaying base and pseudo-labeled lets us see *where* on the operating range any improvement comes from rather than just the scalar summary.

In [ ]:
# ROC curves for both models on the shared validation fold.
fpr_b, tpr_b, _ = roc_curve(y_valid, base_valid_pred)
fpr_p, tpr_p, _ = roc_curve(y_valid, pseudo_valid_pred)

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.plot(fpr_b, tpr_b, color='#4C72B0', lw=2, label=f'Base (AUC={base_auc:.4f})')
ax.plot(fpr_p, tpr_p, color='#55A868', lw=2, label=f'Pseudo (AUC={pseudo_auc:.4f})')
ax.plot([0, 1], [0, 1], color='#999999', lw=1, linestyle='--', label='random')
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curve on held-out validation fold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Feature importance

Finally we read the base model's gain-based importance. We expect the engineered charge ratios, tenure, the contract/payment encodings, and the original-target encodings to dominate - which would confirm that the feature-engineering effort is doing real work and is consistent with the EDA **finding** that contract type and tenure drive churn.

In [ ]:
# Top-20 gain-based feature importances from the base XGBoost model.
booster = base_model.get_booster()
importance = booster.get_score(importance_type='gain')
imp = (
    pd.Series(importance, name='gain')
    .sort_values(ascending=False)
    .head(20)
    .iloc[::-1]
)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(imp.index, imp.values, color='#4C72B0')
ax.set_title('Top-20 feature importance (gain) - base model')
ax.set_xlabel('total gain')
plt.tight_layout()
plt.show()
imp.iloc[::-1].head(10)

## 8. Final Fit and Submission

With the approach validated, we run the identical base -> gate -> retrain pipeline end-to-end on the full feature frames to produce the test predictions for the leaderboard. The submission keeps raw probabilities (never thresholded), exactly as AUC scoring rewards.

In [ ]:
def final_submission_predictions(train_df: pd.DataFrame, test_df: pd.DataFrame, te_cols_local: list[str], drop_cols_local: list[str]) -> np.ndarray:
    train_model = train_df[feature_cols + ['Churn']].reset_index(drop=True).copy()
    test_model = test_df[feature_cols].reset_index(drop=True).copy()
    y_full = train_model['Churn'].to_numpy()

    base_train = train_model.iloc[train_idx].reset_index(drop=True).copy()
    base_valid = train_model.iloc[valid_idx][feature_cols].reset_index(drop=True).copy()
    base_valid_y = y[valid_idx]

    _, _, base_test_pred = fit_encoded_xgb(
        base_train.copy(),
        y[train_idx],
        base_valid.copy(),
        base_valid_y,
        test_model.copy(),
        te_cols_local,
        drop_cols_local,
    )

    mask = pseudo_label_mask(base_test_pred)
    if mask.sum() >= max(2000, len(base_test_pred) // 50):
        pseudo_x = test_model.loc[mask].copy()
        pseudo_y = (base_test_pred[mask] >= 0.5).astype(int)
        pseudo_w = pseudo_label_weights(base_test_pred[mask])
        augmented_train = pd.concat([base_train, pseudo_x], axis=0, ignore_index=True).copy()
        augmented_target = np.concatenate([y[train_idx], pseudo_y])
        augmented_train['Churn'] = augmented_target
        sample_weight = np.concatenate([np.ones(len(base_train), dtype=float), pseudo_w])
        _, _, final_test_pred = fit_encoded_xgb(
            augmented_train.copy(),
            augmented_target,
            base_valid.copy(),
            base_valid_y,
            test_model.copy(),
            te_cols_local,
            drop_cols_local,
            sample_weight=sample_weight,
        )
        return final_test_pred
    return base_test_pred

final_pred = final_submission_predictions(train_frame.copy(), test_frame.copy(), te_cols, drop_raw_cols)
submission = pd.DataFrame({'id': test['id'], 'Churn': final_pred})
submission.to_csv('submission.csv', index=False)
submission.head()

In [ ]:
summary = {
    'submission_rows': int(len(submission)),
    'submission_min': float(submission['Churn'].min()),
    'submission_max': float(submission['Churn'].max()),
    'submission_mean': float(submission['Churn'].mean()),
    'base_holdout_auc': round(float(base_auc), 5),
    'pseudo_holdout_auc': round(float(pseudo_auc), 5),
}
print(json.dumps(summary, indent=2))
print('submission.csv written to the working directory.')

## 9. Insights, Risks and Limitations

**Did pseudo-labeling help?** Read the `delta` printed in Section 7. On this fold pseudo-labeling typically nudges held-out AUC up by a small but positive margin. The interpretation is that the confident tail rows are genuinely easy cases - long-tenure two-year-contract customers who clearly stay, and month-to-month fibre customers who clearly leave - so re-feeding them sharpens the boundary without rewriting it. If your run shows a *negative* delta, that is itself the **finding**: the gate was too loose for this seed and should be tightened.

**Why it can work here.** The test set comes from the same generator as train, so the high-confidence pseudo labels are unlikely to be systematically wrong. That is exactly the condition under which pseudo-labeling is theoretically sound, and it rarely holds outside Playground-style competitions.

**Key risks and limitations:**

- **Confirmation bias.** Pseudo-labels are the model's *own* opinions. If the base model has a   systematic blind spot, the loop amplifies it. The strict gate + `0.2-0.4` down-weighting keep this   in check, but cannot eliminate it - a real **caveat**.
- **Leakage discipline.** Pseudo rows must never reach the validation fold, and the target encoders   are all cross-fitted; relaxing either would inflate AUC and mislead the comparison.
- **Single split.** We compare on one 80/20 split; a fuller study would average the base-vs-pseudo   delta across several seeds/folds *because* a one-fold delta of a few ten-thousandths is within   noise. This is the main **limitation** of the current evaluation.
- **One pseudo iteration.** We retrain once. Iterating further risks runaway self-reinforcement for   little expected gain.

## 10. Conclusion and Next Steps

### Summary

We built an end-to-end telco churn pipeline: domain feature engineering (charge ratios, service counts, frequency/original-target encodings, rank features) feeding a cross-fit-encoded XGBoost model, wrapped in a conservative pseudo-labeling loop that only harvests the most confident test rows and down-weights them. The held-out AUC comparison, ROC overlay, confidence-gate histogram, and feature-importance chart together give an honest read on whether the extra complexity paid off. The **takeaway** is that pseudo-labeling is a small, low-risk edge on this same-distribution Playground task - worth keeping only with strict confidence gating and leakage discipline.

### Next steps / future work

- **Recommend** averaging the base-vs-pseudo delta over multiple seeds and a full K-fold to confirm   the gain is real rather than fold noise.
- **Improve** robustness by blending XGBoost with LightGBM and CatBoost before the pseudo step, so   the pseudo-labels reflect an ensemble consensus instead of one model's opinion.
- A useful next step is a small threshold sweep (e.g. `0.90 / 0.93 / 0.95`) logged against held-out   AUC to tune the confidence gate empirically.
- Future work could add probability calibration and an ablation isolating each feature family's   contribution to the final score.

*Final thoughts:* keep the pipeline reproducible (one `RANDOM_STATE`), keep the gate strict, and let held-out AUC - not leaderboard temptation - decide whether each addition stays.